<a href="https://colab.research.google.com/github/fidlarsyn/Scikit-learn-Cookbook--O-Reilly-/blob/main/11_Novelty_and_Outlier_Detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Introduction to outlier and novelty detection**
Dalam analisis anomali, scikit-learn membedakan data yang tidak biasa menjadi dua kategori utama:

**Outlier Detection (Deteksi Pencilan)**:
* **Kondisi**: Data pelatihan Anda sudah "kotor" dan mengandung observasi yang sangat berbeda dari yang lain.
* **Tujuan**: Mengidentifikasi titik-titik yang menyimpang di dalam kumpulan data yang ada untuk dibersihkan atau dianalisis lebih lanjut.

**Novelty Detection (Deteksi Kebaruan)**:
* **Kondisi**: Data pelatihan dianggap "bersih" (hanya berisi data normal). Model dilatih untuk mengenali apa itu "normal".
* **Tujuan**: Mendeteksi apakah data baru yang masuk (saat tahap prediksi) merupakan sesuatu yang belum pernah dilihat sebelumnya.

# **Understanding Isolation Forest**
Isolation Forest adalah algoritme yang sangat efisien untuk data dimensi tinggi. Alih-alih memprofilkan data normal, algoritme ini bekerja dengan cara mengisolasi anomali.
* **Prinsip Kerja**: Menggunakan pemisahan fitur secara acak. Anomali lebih mudah diisolasi sehingga memiliki jalur pemisahan (path length) yang lebih pendek dalam pohon keputusan.
* **Keunggulan**: Skalabel untuk dataset besar dan tidak bergantung pada metrik jarak atau kepadatan.


**Contoh Kode:**

In [ ]:
from sklearn.ensemble import IsolationForest

# Inisialisasi model dengan estimasi kontaminasi 5%
model = IsolationForest(n_estimators=100, contamination=0.05, random_state=2024)

# Melatih model dan memprediksi (1 = inlier, -1 = outlier)
model.fit(X_combined)
y_pred = model.predict(X_combined)

# **One-Class SVM for novelty detection**
One-Class SVM digunakan ketika Anda hanya memiliki akses ke data normal selama pelatihan. Algoritme ini mempelajari batas "kenormalan" dan menganggap apa pun di luar batas tersebut sebagai sesuatu yang baru (novelty).
* **Parameter Kunci**: nu (mengontrol trade-off antara anomali dan fleksibilitas) dan gamma (mempengaruhi seberapa ketat batas keputusan).

**Contoh Kode:**

In [ ]:
from sklearn.svm import OneClassSVM

# Melatih hanya pada data normal (X_train)
model = OneClassSVM(kernel='rbf', gamma='scale', nu=0.05)
model.fit(X_train)

# Memprediksi pada data baru (X_test)
y_pred = model.predict(X_test)

# **Detecting outliers with LOF**
LOF adalah metode berbasis kepadatan yang membandingkan kepadatan lokal suatu titik dengan tetangganya. Jika suatu titik berada di wilayah yang jauh lebih jarang dibandingkan tetangganya, maka titik tersebut dianggap outlier.
* **Karakteristik**: Sangat efektif untuk dataset di mana kepadatan datanya bervariasi. LOF bersifat unsupervised dan biasanya tidak digunakan untuk prediksi pada data baru yang belum terlihat (unseen data).

**Contoh Kode:**

In [ ]:
from sklearn.neighbors import LocalOutlierFactor

# n_neighbors mengontrol sensitivitas terhadap variasi lokal
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
y_pred = lof.fit_predict(X_with_outliers)

# **Evaluating outlier detection models**
Karena outlier sering kali sangat jarang, metrik standar seperti akurasi tidak cukup. Evaluasi dilakukan menggunakan metrik untuk dataset tidak seimbang:
* **Precision & Recall**: Seberapa akurat model mendeteksi outlier asli tanpa terlalu banyak alarm palsu.
* **ROC-AUC Score**: Mengukur kemampuan model dalam membedakan inlier dan outlier.
* **Confusion Matrix**: Memberikan rincian prediksi benar/salah.

**Contoh Kode:**

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Mengonversi output (-1, 1) ke format biner (0, 1) untuk evaluasi
y_pred_binary = np.where(y_pred == 1, 0, 1)
print(classification_report(y_true, y_pred_binary))

# **Handling detected outliers**
Setelah terdeteksi, ada beberapa strategi untuk menangani anomali tersebut:
* **Removal (Penghapusan)**: Menghapus baris outlier jika dianggap sebagai kesalahan sensor atau data rusak.
* **Replacement (Penggantian)**: Mengganti nilai outlier dengan statistik pusat seperti median atau mean agar ukuran dataset tetap sama.
* **Capping (Winsorization)**: Membatasi nilai ekstrem ke batas persentil tertentu (misal: persentil ke-5 atau ke-95).

**Contoh Kode:**

In [ ]:
# 1. Penghapusan
X_cleaned = X[~outlier_mask]

# 2. Penggantian dengan Median
median_vals = np.median(X[~outlier_mask], axis=0)
X_replaced[outlier_mask] = median_vals

# 3. Capping (Winsorization) menggunakan Clip
X_capped = X_df.clip(lower=X_df.quantile(0.05), upper=X_df.quantile(0.95), axis=1)